# 10 — Holdout v4 + Dekontaminasi Berkode

Penutup Fase 1 untuk issue #4. CPU murni, stdlib + sympy (opsional) + matplotlib.

Yang dikerjakan:

1. **Split holdout 600** (300 numglue + 300 easy) dari `*_clean_v4.jsonl`, sisanya `train_pool`.
2. **Tiga utang verifikasi** dijawab tertulis di dalam notebook ini:
   - V1 — kebocoran *bare-letter* (`^[A-E]$`) ke holdout lama.
   - V2 — `antlr4-python3-runtime` terpasang atau tidak (menentukan apakah
     `sympy.parse_latex` hidup; kalau mati, holdout bias ke jawaban numerik murni).
   - V3 — dekontaminasi jadi **kode**, bukan konvensi manual: overlap persis DAN
     near-duplicate (Jaccard >= 0.5).

> ## BELUM DIJALANKAN
>
> Notebook ini ter-commit **tanpa output**. Prasyaratnya `data/Final/easy_clean_v4.jsonl`
> dan `numglue_clean_v4.jsonl` — keluaran issue #3, yang **belum ada di repo saat commit ini
> dibuat**. Sel PRA-TERBANG di bawah gagal keras kalau file itu belum ada.
>
> Angka target dari dry-run v3 (holdout 600, train_pool 5.416 = numglue 3.462 + easy 1.954,
> dekontaminasi buang 0) adalah **acuan, bukan hasil**. Hitung ulang setelah v4.

Catatan penyimpanan: `data/*` masuk `.gitignore` kecuali `data/Final/*_v3.*`, jadi
holdout/train_pool v4 **tidak** ikut ter-track. Yang bertahan di repo adalah laporan
JSON di `reports/10_holdout_dekontaminasi.json` — di situlah angka untuk paper diambil.

## 0. Bootstrap

In [ ]:
import os, sys
p = os.getcwd()
while not os.path.isdir(os.path.join(p, 'src')) and os.path.dirname(p) != p:
    p = os.path.dirname(p)
os.chdir(p); sys.path.insert(0, p)

import json
import random
import re
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt

print('repo root:', p)

## 1. Konfigurasi

`SEED` dan `N_PER_SUBSET` menentukan isi holdout. Sekali dijalankan dan angkanya masuk paper,
**jangan diubah** — mengubahnya berarti seluruh tabel evaluasi hilang basisnya.

`HOLDOUT_LAMA` menunjuk file holdout 600 versi lama (ada di Kaggle/Drive, tidak di repo).
Diperlukan hanya untuk verifikasi V1. Biarkan `None` kalau file belum diunduh — sel V1
akan melapor "tidak dapat diverifikasi", bukan diam-diam lolos.

In [ ]:
SEED = 42
N_PER_SUBSET = 300                     # 300 numglue + 300 easy = holdout 600

SUMBER = {
    'numglue': Path('data/Final/numglue_clean_v4.jsonl'),
    'easy':    Path('data/Final/easy_clean_v4.jsonl'),
}

OUT_HOLDOUT    = Path('data/Final/holdout_v4.jsonl')
OUT_TRAIN_POOL = Path('data/Final/train_pool_v4.jsonl')
OUT_LAPORAN    = Path('reports/10_holdout_dekontaminasi.json')

# Holdout 600 versi LAMA, untuk verifikasi V1. Isi path lokalnya kalau sudah diunduh.
HOLDOUT_LAMA = None                    # mis. Path('data/eval/holdout_v2.jsonl')

AMBANG_JACCARD = 0.5                   # near-duplicate: Jaccard token >= ambang

laporan = {'konfigurasi': {'seed': SEED, 'n_per_subset': N_PER_SUBSET,
                           'ambang_jaccard': AMBANG_JACCARD}}
print(json.dumps(laporan['konfigurasi'], indent=2))

## 2. PRA-TERBANG

Gagal keras kalau input v4 belum ada. Itu tujuannya: menjalankan split di atas v3 lalu
melaporkan angkanya sebagai "v4" adalah cara paling murah merusak seluruh Fase 2.

In [ ]:
kurang = [str(v) for v in SUMBER.values() if not v.exists()]
if kurang:
    raise FileNotFoundError(
        'Prasyarat issue #3 belum ada: ' + ', '.join(kurang) +
        '\nJangan ganti ke v3 lalu melanjutkan. Tunggu branch issue-3 di-merge.'
    )

def muat(path: Path) -> list[dict]:
    """Baca JSONL jadi list dict."""
    return [json.loads(l) for l in open(path, encoding='utf-8') if l.strip()]

data = {nama: muat(path) for nama, path in SUMBER.items()}
laporan['input'] = {nama: len(rows) for nama, rows in data.items()}
for nama, rows in data.items():
    print(f'{nama:10}: {len(rows):>6} baris  <- {SUMBER[nama]}')
print('total     :', sum(len(r) for r in data.values()))

## 3. Verifikasi V1 — kebocoran *bare-letter* ke holdout lama

**Akar masalah.** `src/eval/make_holdout.py:36` memakai `re.findall(r"[A-Za-z]{3,}", j)`;
huruf tunggal tidak pernah match, sehingga jawaban `"C"` dihitung sebagai `single_expr`
-> masuk himpunan GRADEABLE. Lalu `src/eval/clean_holdout.py:33` membiarkannya lolos
karena sympy mem-parse `"C"` sebagai simbol matematika yang sah.

Akibatnya soal pilihan-ganda berjawaban huruf bisa duduk di holdout dan dinilai seolah
jawaban numerik. Sel ini menghitung berapa banyak yang benar-benar bocor.

In [ ]:
BARE_LETTER = re.compile(r'^[A-E]$')

def cek_bare_letter(rows: list[dict]) -> dict:
    """Hitung baris berjawaban huruf tunggal A-E (kebocoran pilihan ganda)."""
    bocor = [r for r in rows
             if BARE_LETTER.match((r.get('jawaban') or '').strip().strip('.').strip('$').strip())]
    return {'total': len(rows), 'bocor': len(bocor),
            'rasio': round(len(bocor) / len(rows), 4) if rows else 0.0,
            'contoh': [r.get('soal', '')[:120] for r in bocor[:5]]}

if HOLDOUT_LAMA is None or not Path(HOLDOUT_LAMA).exists():
    v1 = {'status': 'TIDAK DAPAT DIVERIFIKASI',
          'sebab': 'HOLDOUT_LAMA belum diisi / file tidak ada di mesin ini',
          'tindakan': 'unduh holdout 600 lama dari Kaggle/Drive, isi HOLDOUT_LAMA, jalankan ulang sel ini'}
else:
    v1 = {'status': 'DIVERIFIKASI', 'file': str(HOLDOUT_LAMA)}
    v1.update(cek_bare_letter(muat(Path(HOLDOUT_LAMA))))

laporan['verifikasi_1_bare_letter_holdout_lama'] = v1
print(json.dumps(v1, ensure_ascii=False, indent=2))

## 4. Verifikasi V2 — `antlr4-python3-runtime`

`sympy.parsing.latex.parse_latex` memerlukan runtime ANTLR. Tanpa paket itu, **semua**
jawaban LaTeX gagal di-parse -> dianggap tak-gradeable -> holdout bias ke jawaban numerik
murni, dan klaim "198/300 gold easy berupa ekspresi simbolik" (Tabel V) tidak mungkin benar.

Bukti tak langsung tidak cukup. Sel ini mencobanya langsung: satu string LaTeX sederhana.
Kalau gagal, penyebab paling mungkin ANTLR tak terpasang.

In [ ]:
v2 = {}
try:
    import antlr4                                     # noqa: F401
    v2['antlr4_terpasang'] = True
    v2['antlr4_versi'] = getattr(antlr4, '__version__', 'tidak diketahui')
except Exception as e:
    v2['antlr4_terpasang'] = False
    v2['antlr4_error'] = f'{type(e).__name__}: {e}'

try:
    import sympy
    from sympy.parsing.latex import parse_latex
    v2['sympy_versi'] = sympy.__version__
    try:
        parse_latex(r'\frac{1}{2}')
        v2['parse_latex_hidup'] = True
    except Exception as e:
        v2['parse_latex_hidup'] = False
        v2['parse_latex_error'] = f'{type(e).__name__}: {str(e)[:200]}'
except Exception as e:
    v2['sympy_versi'] = None
    v2['parse_latex_hidup'] = False
    v2['parse_latex_error'] = f'{type(e).__name__}: {e}'

v2['dampak'] = ('AMAN: gold LaTeX bisa dinilai' if v2.get('parse_latex_hidup')
                else 'BIAS: gold LaTeX dianggap tak-gradeable -> holdout condong ke angka murni. '
                     'Pasang `pip install antlr4-python3-runtime==4.11` lalu jalankan ulang notebook.')

laporan['verifikasi_2_antlr4'] = v2
print(json.dumps(v2, ensure_ascii=False, indent=2))

## 5. Klasifikasi jawaban — dengan bug V1 sudah ditambal

`answer_type` di bawah adalah salinan `make_holdout.py` **plus** cabang `bare_letter`
yang dipasang sebelum cabang `single_expr`. Cabang itulah tambalan bug V1.

`bare_letter` sengaja tidak dimasukkan ke `GRADEABLE`: soal pilihan ganda tetap boleh
hidup di `train_pool`, hanya dilarang masuk holdout.

In [ ]:
INT = re.compile(r'^-?\d+$')
NUM = re.compile(r'^-?\d+([./]\d+)?$')
GRADEABLE = {'pure_int', 'pure_num', 'single_expr'}

def answer_type(jawaban: str) -> str:
    """Klasifikasi bentuk jawaban. `bare_letter` = tambalan bug make_holdout.py:36."""
    j = (jawaban or '').strip()
    if not j:
        return 'empty'
    core = j.strip('.').strip('$').strip()
    if BARE_LETTER.match(core):
        return 'bare_letter'
    if INT.match(core):
        return 'pure_int'
    if NUM.match(core):
        return 'pure_num'
    if len(re.findall(r'[A-Za-z]{3,}', j)) <= 1:
        return 'single_expr'
    return 'sentence' if ('$' in j or '\\' in j) else 'sentence_plain'


def latex_seimbang(s: str) -> bool:
    """Delimiter LaTeX berpasangan. Soal timpang tak terbaca -> tak layak holdout."""
    s = s or ''
    return (s.count(r'\(') == s.count(r'\)')
            and s.count(r'\[') == s.count(r'\]')
            and s.count('{') == s.count('}')
            and s.count('$') % 2 == 0)


def gold_terparse(g: str) -> bool:
    """Gold bisa dinilai otomatis: angka murni, atau bisa diparse sympy."""
    core = (g or '').strip().strip('.').strip('$').strip()
    if INT.match(core) or NUM.match(core):
        return True
    try:
        from src.eval.answer_check import _to_expr
        return _to_expr(g) is not None
    except Exception:
        return False


for nama, rows in data.items():
    for r in rows:
        r['answer_type'] = answer_type(r.get('jawaban', ''))
    print(nama, dict(Counter(r['answer_type'] for r in rows).most_common()))

## 6. Split holdout 600 + train_pool

Kelayakan holdout = tiga syarat, semuanya wajib:

1. `answer_type` ada di `GRADEABLE` (sudah menutup `bare_letter`),
2. soal LaTeX-nya seimbang,
3. gold-nya benar-benar bisa diparse (numerik atau sympy).

Syarat 3 bergantung pada V2. Kalau `parse_latex` mati, kolam layak menciut dan condong
ke angka murni — komposisi di sel berikutnya akan memperlihatkannya.

In [ ]:
holdout, train_pool, alasan_tolak = [], [], defaultdict(Counter)

for nama, rows in data.items():
    layak, tak_layak = [], []
    for r in rows:
        if r['answer_type'] not in GRADEABLE:
            alasan_tolak[nama][f"answer_type={r['answer_type']}"] += 1
            tak_layak.append(r); continue
        if not latex_seimbang(r.get('soal', '')):
            alasan_tolak[nama]['latex_timpang'] += 1
            tak_layak.append(r); continue
        if not gold_terparse(r.get('jawaban', '')):
            alasan_tolak[nama]['gold_tak_terparse'] += 1
            tak_layak.append(r); continue
        layak.append(r)

    random.Random(SEED).shuffle(layak)
    n = min(N_PER_SUBSET, len(layak))
    if n < N_PER_SUBSET:
        print(f'PERINGATAN {nama}: kolam layak hanya {len(layak)}, diminta {N_PER_SUBSET}')
    for r in layak[:n]:
        r['subset'] = nama
        holdout.append(r)
    for r in layak[n:] + tak_layak:
        r['subset'] = nama
        train_pool.append(r)

laporan['split'] = {
    'holdout_total': len(holdout),
    'holdout_per_subset': dict(Counter(r['subset'] for r in holdout)),
    'holdout_komposisi_answer_type': dict(Counter(r['answer_type'] for r in holdout)),
    'train_pool_total': len(train_pool),
    'train_pool_per_subset': dict(Counter(r['subset'] for r in train_pool)),
    'alasan_tak_layak_holdout': {k: dict(v) for k, v in alasan_tolak.items()},
}
print(json.dumps(laporan['split'], ensure_ascii=False, indent=2))

## 7. Verifikasi V3 — dekontaminasi sebagai kode

Temuan sebelumnya: kolam teacher-correct lama memuat **127/300** soal test numglue dan
**194/300** easy. SFT final kebetulan bersih (0 tumpang tindih persis), tapi **tak ada kode
yang menjaminnya**. Fungsi di bawah adalah jaminannya.

Dua lapis:

- **overlap persis** — soal dinormalkan (lowercase, tanda baca dibuang, spasi dirapatkan),
  lalu dicocokkan lewat himpunan hash. O(n+m).
- **near-duplicate** — Jaccard token >= `AMBANG_JACCARD`. Dipercepat dengan indeks terbalik:
  hanya kandidat yang berbagi minimal satu token yang dihitung, bukan seluruh perkalian silang.
  Sebelumnya 9/300 (numglue) dan 2/300 (easy).

`dekontaminasi()` sengaja generik: dipakai di sini untuk `train_pool` vs `holdout`, dan
dipakai lagi di issue #9/#10 untuk `correct_<teacher>.jsonl` vs holdout.

In [ ]:
_PUNCT = re.compile(r'[^\w\s]', re.UNICODE)
_WS = re.compile(r'\s+')

def normalisasi_soal(s: str) -> str:
    """Bentuk kanonik untuk pencocokan persis."""
    return _WS.sub(' ', _PUNCT.sub(' ', (s or '').lower())).strip()


def token_set(s: str) -> frozenset:
    return frozenset(normalisasi_soal(s).split())


def jaccard(a: frozenset, b: frozenset) -> float:
    if not a or not b:
        return 0.0
    inter = len(a & b)
    return inter / (len(a) + len(b) - inter)


def dekontaminasi(pool: list[dict], acuan: list[dict], ambang: float = 0.5,
                  kunci: str = 'soal') -> tuple[list[dict], dict]:
    """Buang baris `pool` yang bocor dari `acuan`.

    Mengembalikan (pool_bersih, laporan). Dua lapis: overlap persis (hash pada
    bentuk ternormalisasi) lalu near-duplicate (Jaccard token >= ambang, dipercepat
    indeks terbalik). Baris yang dibuang dicatat beserta pasangan acuannya supaya
    keputusannya bisa diaudit, bukan dipercaya begitu saja.
    """
    acuan_norm = {normalisasi_soal(r.get(kunci, '')) for r in acuan}
    acuan_tok = [token_set(r.get(kunci, '')) for r in acuan]

    indeks = defaultdict(list)                      # token -> indeks baris acuan
    for i, ts in enumerate(acuan_tok):
        for t in ts:
            indeks[t].append(i)

    bersih, buang_persis, buang_neardup = [], [], []
    for r in pool:
        s = r.get(kunci, '')
        if normalisasi_soal(s) in acuan_norm:
            buang_persis.append(r)
            continue
        ts = token_set(s)
        kandidat = {i for t in ts for i in indeks.get(t, ())}
        skor_max, pasangan = 0.0, None
        for i in kandidat:
            sk = jaccard(ts, acuan_tok[i])
            if sk > skor_max:
                skor_max, pasangan = sk, i
        if skor_max >= ambang:
            buang_neardup.append({'soal': s[:160], 'jaccard': round(skor_max, 3),
                                  'acuan': acuan[pasangan].get(kunci, '')[:160]})
            continue
        bersih.append(r)

    lap = {
        'pool_masuk': len(pool),
        'acuan': len(acuan),
        'ambang_jaccard': ambang,
        'overlap_persis': len(buang_persis),
        'near_duplicate': len(buang_neardup),
        'total_dibuang': len(buang_persis) + len(buang_neardup),
        'pool_bersih': len(bersih),
        'contoh_overlap_persis': [r.get(kunci, '')[:160] for r in buang_persis[:5]],
        'contoh_near_duplicate': buang_neardup[:5],
    }
    return bersih, lap


# --- swa-uji: kontaminasi buatan, harus tertangkap keduanya ---
_acuan_uji = [{'soal': 'Berapa hasil dari dua tambah tiga?'}]
_pool_uji = [
    {'soal': 'berapa hasil dari dua tambah tiga'},                    # persis (setelah normalisasi)
    {'soal': 'Berapa hasil dari dua tambah tiga saja'},               # near-dup
    {'soal': 'Luas lingkaran berjari-jari tujuh sentimeter'},         # bersih
]
_bersih_uji, _lap_uji = dekontaminasi(_pool_uji, _acuan_uji, AMBANG_JACCARD)
assert _lap_uji['overlap_persis'] == 1, _lap_uji
assert _lap_uji['near_duplicate'] == 1, _lap_uji
assert _lap_uji['pool_bersih'] == 1, _lap_uji
print('swa-uji dekontaminasi LOLOS:', {k: _lap_uji[k] for k in
      ('overlap_persis', 'near_duplicate', 'pool_bersih')})

In [ ]:
train_pool_bersih, lap_dekon = dekontaminasi(train_pool, holdout, AMBANG_JACCARD)

# Rincian per subset supaya angka 127/300 & 194/300 lama punya pembanding langsung.
per_subset = {}
for nama in SUMBER:
    sub_hold = [r for r in holdout if r['subset'] == nama]
    sub_pool = [r for r in train_pool if r['subset'] == nama]
    _, lap_sub = dekontaminasi(sub_pool, sub_hold, AMBANG_JACCARD)
    per_subset[nama] = {k: lap_sub[k] for k in
                        ('pool_masuk', 'overlap_persis', 'near_duplicate', 'pool_bersih')}

laporan['verifikasi_3_dekontaminasi'] = {'gabungan': lap_dekon, 'per_subset': per_subset}
print(json.dumps(laporan['verifikasi_3_dekontaminasi'], ensure_ascii=False, indent=2)[:3000])

### 7b. Sensitivitas ambang — ambang 0.5 kemungkinan besar terlalu longgar

Dry-run di atas data v3 (seed 42, holdout 600 yang sama) memberi **452** near-duplicate
pada ambang 0.5, jauh di atas angka lama 9/300 dan 2/300. Contoh yang tercetak
memperlihatkan sebabnya: numglue penuh soal bertemplat.

```
"Carilah jumlah mol Hidrogen ... 3 mol Asam Sulfat dan 3 mol Seng"
vs "Carilah jumlah mol NaCl ... 3 mol HCl dan 3 mol NaHCO3"       Jaccard 0.562
```

Kalimatnya kembar, **soalnya beda** — zat berbeda, jawaban berbeda. Itu positif palsu.
Jaccard token tidak melihat entitas yang justru membawa isi soal.

Karena itu ambang tidak boleh dipilih dari kebiasaan. Sel di bawah menyapu 0.5–0.9 dan
melaporkan jumlah buangan per ambang. Aturan pakai:

- **laporkan angka ambang 0.5** di paper (itu yang diminta issue, dan itu batas atas
  kontaminasi yang paling konservatif),
- **pilih ambang operasional** untuk `train_pool` dari sapuan ini plus inspeksi manual
  contoh di sekitar ambang. Tulis alasannya di komentar issue, jangan diam-diam.

In [ ]:
sapuan = {}
for amb in (0.5, 0.6, 0.7, 0.8, 0.9):
    _, lap_a = dekontaminasi(train_pool, holdout, amb)
    sapuan[amb] = {k: lap_a[k] for k in ('overlap_persis', 'near_duplicate',
                                         'total_dibuang', 'pool_bersih')}
    print(f'ambang {amb}: near-dup {lap_a["near_duplicate"]:>5} | '
          f'total buang {lap_a["total_dibuang"]:>5} | sisa {lap_a["pool_bersih"]:>5}')

laporan['verifikasi_3_dekontaminasi']['sapuan_ambang'] = sapuan

# Contoh tepat di pita 0.5-0.6: inspeksi manual, tentukan positif palsu atau bukan.
print('\n--- contoh pita 0.5-0.6 (periksa manual) ---')
for c in lap_dekon['contoh_near_duplicate']:
    if 0.5 <= c['jaccard'] < 0.6:
        print(f"[{c['jaccard']}] pool : {c['soal']}")
        print(f"        acuan: {c['acuan']}\n")

## 8. Tulis keluaran

In [ ]:
def tulis(rows: list[dict], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

tulis(holdout, OUT_HOLDOUT)
tulis(train_pool_bersih, OUT_TRAIN_POOL)

laporan['keluaran'] = {
    'holdout_path': str(OUT_HOLDOUT), 'holdout_baris': len(holdout),
    'train_pool_path': str(OUT_TRAIN_POOL), 'train_pool_baris': len(train_pool_bersih),
}

OUT_LAPORAN.parent.mkdir(parents=True, exist_ok=True)
with open(OUT_LAPORAN, 'w', encoding='utf-8') as f:
    json.dump(laporan, f, ensure_ascii=False, indent=2)

print(json.dumps(laporan['keluaran'], ensure_ascii=False, indent=2))
print('laporan ->', OUT_LAPORAN, '(ini yang ter-track git, bukan data/)')

## 9. Visualisasi — komposisi holdout & sebab penolakan

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))

komp = Counter(r['answer_type'] for r in holdout)
ax[0].bar(list(komp.keys()), list(komp.values()), color='#4C72B0')
ax[0].set_title(f'Komposisi holdout (n={len(holdout)})')
ax[0].set_ylabel('jumlah soal')
ax[0].tick_params(axis='x', rotation=20)
for i, v in enumerate(komp.values()):
    ax[0].text(i, v, str(v), ha='center', va='bottom', fontsize=9)

tolak = Counter()
for sub in alasan_tolak.values():
    tolak.update(sub)
label = [k for k, _ in tolak.most_common(8)][::-1]
nilai = [tolak[k] for k in label]
ax[1].barh(label, nilai, color='#C44E52')
ax[1].set_title('Sebab tak layak holdout (kedua subset)')
ax[1].set_xlabel('jumlah soal')
for i, v in enumerate(nilai):
    ax[1].text(v, i, ' ' + str(v), va='center', fontsize=9)

plt.tight_layout()
plt.show()

## 10. Ringkasan untuk komentar issue #4

Sel di bawah mencetak blok siap-tempel. Tempel apa adanya ke thread issue — angkanya
itulah memori agen berikutnya (issue #5). Jangan rapikan angka yang tidak enak dilihat.

In [ ]:
v1 = laporan['verifikasi_1_bare_letter_holdout_lama']
v2 = laporan['verifikasi_2_antlr4']
v3 = laporan['verifikasi_3_dekontaminasi']['gabungan']
sp = laporan['split']
kel = laporan['keluaran']

print(f"""### Hasil issue #4 - holdout v4 + dekontaminasi

**Split**
- holdout: {sp['holdout_total']} ({sp['holdout_per_subset']})
- train_pool setelah dekontaminasi: {kel['train_pool_baris']}
- komposisi answer_type holdout: {sp['holdout_komposisi_answer_type']}
- seed={SEED}, n_per_subset={N_PER_SUBSET} -- BEKU, jangan diubah lagi

**V1 bare-letter di holdout lama**
- status: {v1['status']}
- bocor: {v1.get('bocor', '-')} / {v1.get('total', '-')}

**V2 antlr4-python3-runtime**
- terpasang: {v2['antlr4_terpasang']} | parse_latex hidup: {v2.get('parse_latex_hidup')}
- dampak: {v2['dampak']}

**V3 dekontaminasi (berkode, bukan konvensi)**
- overlap persis: {v3['overlap_persis']}
- near-duplicate (Jaccard >= {AMBANG_JACCARD}): {v3['near_duplicate']}
- total dibuang: {v3['total_dibuang']} dari {v3['pool_masuk']}
- per subset: {laporan['verifikasi_3_dekontaminasi']['per_subset']}
- sapuan ambang: {laporan['verifikasi_3_dekontaminasi'].get('sapuan_ambang')}
- CATATAN: ambang 0.5 over-fire pada soal numglue bertemplat (positif palsu). Angka 0.5
  dilaporkan sebagai batas atas; ambang operasional dipilih dari sapuan + inspeksi manual.

**File**
- {kel['holdout_path']} (gitignored)
- {kel['train_pool_path']} (gitignored)
- {OUT_LAPORAN} (ter-track -- sumber angka paper)

**Untuk agen #5**: data Fase 1 BEKU per sini. Fungsi `dekontaminasi()` ada di sel 7
notebook ini; pakai ulang untuk menyaring `correct_<teacher>.jsonl` vs holdout sebelum
menghitung skor teacher apa pun.""")